# Public-land reconciliation audit — 2026-09-22

This companion recomputes key published counts and discrepancy summaries from the locally cached records. It does not fetch data or change candidate selection. Complete spatial methods and exact source requests are in `fetch-bcplm-reference.py`, `audit-public-land.py`, and `audit-spatial-followups.py`. The snapshot report is `output/public-land/audit-report.md`.

The unit of comparison is a published row, a distinct PID, or a current physical parcel as explicitly labelled. The data have different snapshot dates; spatial differences do not establish historical title changes.

In [1]:
from pathlib import Path
from collections import Counter
import csv, gzip, json
ROOT = Path.cwd()
assert (ROOT / "source/public-land/ubc-published-geometry.json.gz").exists()
CACHE = ROOT / "source/public-land"
OUT = ROOT / "output/public-land"
with gzip.open(CACHE / "ubc-published-geometry.json.gz", "rt") as stream:
    published = json.load(stream)
rows = [feature["properties"] for feature in published["features"]]
summary = json.loads((OUT / "audit-summary.json").read_text())
followups = json.loads((OUT / "audit-followups.json").read_text())
assert len(rows) == len({row["OBJECTID"] for row in rows}) == published["metadata"]["count"]
print(published["metadata"])

{'url': 'https://services2.arcgis.com/NlsizNmbMFiinWw4/arcgis/rest/services/BCPLM_Apr13/FeatureServer/214', 'editingInfo': {'lastEditDate': 1787768010465, 'schemaLastEditDate': 1787768007974, 'dataLastEditDate': 1787762105272}, 'queriedAt': '2026-09-22T19:44:30Z', 'count': 52384}


In [2]:
pids = [str(int(row["PID_NUMBER"])).zfill(9) for row in rows]
counts = Counter(pids)
keys = Counter((pid, row["JURISDICTION_CODE"], row["ROLL_NUMBER"]) for pid, row in zip(pids, rows))
profile = {"published_rows": len(rows), "distinct_pids": len(counts),
           "repeated_pids": sum(n > 1 for n in counts.values()),
           "extra_rows_beyond_pid": sum(n - 1 for n in counts.values()),
           "extra_rows_beyond_pid_jurisdiction_roll": sum(n - 1 for n in keys.values())}
assert profile["distinct_pids"] == summary["published_distinct_pids"]
assert profile["extra_rows_beyond_pid_jurisdiction_roll"] == summary["duplicate_pids"]["extra_rows_beyond_pid_jurisdiction_roll_key"]
print(profile)

{'published_rows': 52384, 'distinct_pids': 50100, 'repeated_pids': 2173, 'extra_rows_beyond_pid': 2284, 'extra_rows_beyond_pid_jurisdiction_roll': 1915}


In [3]:
areas = [row["Area_m2"] for row in rows]
area_counts = {"below_100m2": sum(a < 100 for a in areas),
               "100m2_to_2ha": sum(100 <= a <= 20000 for a in areas),
               "above_2ha": sum(a > 20000 for a in areas), "missing_area": 0}
assert area_counts == summary["published_reported_area"]
assert area_counts["above_2ha"] == followups["published_footprint_checks"]["published_geometry_area_above_2ha"]
print(area_counts)
print(followups["diagnostic_candidate_filters"])

{'below_100m2': 0, '100m2_to_2ha': 48857, 'above_2ha': 3527, 'missing_area': 0}
{'retained': 304646, 'with_pid': 219586, 'within_100m2_to_2ha': 192536, 'with_pid_and_within_100m2_to_2ha': 152909, 'applied_to_baseline': False}


In [4]:
with (CACHE / "audit-exceptions.csv").open() as stream:
    exceptions = list(csv.DictReader(stream))
spatial = [r for r in exceptions if r["status"] == "excluded_spatially"]
checks = {"spatial_exception_rows": len(spatial),
          "current_overlap_under_1m2": sum(float(r["current_overlap_m2"]) < 1 for r in spatial),
          "current_overlap_under_1pct": sum(float(r["current_overlap_fraction"]) < .01 for r in spatial),
          "current_overlap_at_least_50pct": sum(float(r["current_overlap_fraction"]) >= .5 for r in spatial),
          "published_overlap_at_least_50pct": sum(float(r["published_overlap_fraction"]) >= .5 for r in spatial)}
assert checks["spatial_exception_rows"] == summary["classification"]["excluded_spatially"]
assert checks["current_overlap_at_least_50pct"] == summary["spatial_exceptions"]["current_overlap_at_least_50pct"]
assert checks["published_overlap_at_least_50pct"] == followups["published_footprint_checks"]["spatial_exceptions_published_overlap_at_least_50pct"]
print(checks)

{'spatial_exception_rows': 457, 'current_overlap_under_1m2': 54, 'current_overlap_under_1pct': 153, 'current_overlap_at_least_50pct': 215, 'published_overlap_at_least_50pct': 216}


In [5]:
print("Region check:", followups["regional_spatial_check"])
print("Current fabric check:", followups["fabric_missing_pid_check"])
print("Geometry comparison:", summary["geometry_comparison"])
print("Ownership disagreements:", summary["ownership_disagreements"])
assert followups["fabric_missing_pid_check"]["positive_control_found"]
assert sum(summary["classification"].values()) == len(rows)
assert sum(summary["published_reported_area"].values()) == len(rows)

Region check: {'published_label_disagrees_with_spatial_rd': 6972, 'current_label_agrees_where_published_disagrees': 6890, 'no_rd_polygon_at_published_interior_point': 161}
Current fabric check: {'requested_missing_pids': 144, 'found_missing_pids': 0, 'positive_control_pid': '024336742', 'positive_control_found': True, 'response_timestamp': '2026-09-22T19:50:10.060Z', 'layer': 'WHSE_CADASTRE.PMBC_PARCEL_FABRIC_POLY_SVW'}
Geometry comparison: {'compared_records': 52240, 'invalid_published_repaired_for_audit': 3, 'invalid_current_repaired_for_audit': 0, 'iou_at_least_0_999': 41297, 'iou_at_least_0_99': 49109, 'iou_below_0_95': 2411, 'iou_below_0_5': 1612, 'median_iou': 0.9999999998652971, 'minimum_iou': 0.0, 'published_area_covered_at_least_99pct': 49681}
Ownership disagreements: [{'published': 'Crown provincial', 'current': 'Private', 'records': 45}, {'published': 'Local government', 'current': 'Private', 'records': 33}, {'published': 'Local government', 'current': 'Crown Provincial', 'r

## Interpretation limits
- The current masks cannot prove what was inside a reserve or park in January 2026.
- PID multiplicity does not automatically mean strata or erroneous duplicate records.
- Regional placement uses an interior point; boundary-spanning parcels may require an area-based assignment.
- Threshold checks are diagnostics, not newly applied eligibility rules.
- Current ownership categories and WHEN_UPDATED do not establish dated ownership transfers.
- No missing PID matched the current fabric query, but this does not establish historical strata membership.